# Optimize enhancer specificity with Malinois

[Notebook](enhancer.ipynb) · [Installation](../docs/installation.md)

Design 200-nt enhancers for K562. The Malinois predictor adds two fixed 200-nt assay flanks, and its network scores the resulting 600-nt construct. The objective subtracts the higher off-target prediction from the K562 prediction.

## 1. Load the model and define the objective



In [1]:
# %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"

import os
from pathlib import Path

import torch
from bioseq_models import load_model
from MotifSift import call_motifs, generate_designs, random_seed_sequences, select_candidates

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
predictor = load_model("malinois", device=device)
# Extract the fixed flanks from a placeholder construct.
reference = predictor.encoder(["A" * 200]).transpose(1, 2)
left, right = reference[:, :200], reference[:, -200:]


class EnhancerScorer:
    task_names = predictor.task_names
    device = predictor.device

    def score(self, sequences):
        return predictor.score(sequences)

    def score_soft(self, probabilities):
        n = len(probabilities)
        construct = torch.cat([
            left.to(probabilities).expand(n, -1, -1),
            probabilities,
            right.to(probabilities).expand(n, -1, -1),
        ], dim=1)
        return predictor.model(construct.transpose(1, 2))


model = EnhancerScorer()


def specificity(y):
    return y["K562_log2FC"] - torch.maximum(y["HepG2_log2FC"], y["SKNSH_log2FC"])


run = Path("outputs/malinois_k562")
os.makedirs(run, exist_ok=True)




The scoring adapter adds the same fixed flanks to soft inputs as the pretrained predictor uses for sequence strings.

## 2. Generate candidates


The saved tables are from a full rerun with base seed 42 and FP32 inference. That run used the assay’s archived GPU batch sizes (1,024 discrete and 128 differentiable).


In [2]:
seeds = random_seed_sequences(200, 100, seed=42)
designs = generate_designs(
    model, length=200, initial=seeds, objective=specificity,
    output_dir=run / "designs",
)
designs.groupby("algorithm")["score"].agg(["size", "median", "max"])


,size,median,max
algorithm,,,
adalead,100,25.317342,27.155134
basesearch,100,23.795057,25.563915
beam_search,100,27.427982,30.767609
cross_entropy_method,100,28.943089,29.598214
den,100,23.908835,25.174662
fast_seqprop,100,26.637044,28.835720
genetic_algorithm,100,33.412852,34.887356
ism_greedy,100,28.653624,31.538803
ledidi,100,24.768156,26.851630




## 3. Discover motifs

Discover motifs from nucleotide contributions to the specificity objective.



In [3]:
motifs = call_motifs(designs, model, output_dir=run / "motifs")
motifs.support.query("retained")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]


,motif_id,seqlets,parents,sites_on_unique_DNA
0,1-CCTTATCTB,3020,1057,5517.0
1,2-AMCAKMTGKT,487,411,1345.0
2,3-VTGASTCAB,310,243,796.0
3,4-WTTCCWGGAAW,54,45,768.0
4,5-ATGCCTTCCG,52,52,103.0
5,7-CGAKAATYKAC,56,56,126.0
7,13-TCATGACTAA,86,86,94.0
10,17-ACCACACC,40,40,191.0




## 4. Select a panel

Use the 20th percentile of the unique candidates’ scores as the minimum score for the panel.



In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
panel = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
panel[["selection_rank", "sequence", "score", "present_motifs"]]


,selection_rank,sequence,score,present_motifs
0,1,GTTATCAAACCGCTGTGTCTGGTTCGTACAGATAACGCTGACATTT...,34.887356,1-CCTTATCTB;2-AMCAKMTGKT;3-VTGASTCAB;4-WTTCCWG...
1,2,GATGTGGCGGTTAGATAAGTTAAATGTGGCTGTCAACGATAGGCCT...,28.800381,1-CCTTATCTB;17-ACCACACC
2,3,GGGAGTCACCTTATCAATGGGAACACATGTTGCAGAGTCATTCATG...,25.046870,1-CCTTATCTB;2-AMCAKMTGKT;5-ATGCCTTCCG;7-CGAKAA...
3,4,TCTTGCGCGAGGGTGCCTTATCTGTTCCTTGCAGCACTGTGGGGTT...,28.136970,1-CCTTATCTB;4-WTTCCWGGAAW;5-ATGCCTTCCG
4,5,GTTGGGGGCTGTCAAAAGTCGCTCTTATCTGGTGGCAACAGCTGTT...,30.207198,1-CCTTATCTB;2-AMCAKMTGKT;7-CGAKAATYKAC
5,6,GCTGACTCAGGGCGTGGTATCGCTACTTCTGCATATGTGAGATAAG...,24.067966,1-CCTTATCTB;3-VTGASTCAB
6,7,TTGATAACCGTTTTTGGCCACTCCTGCCTGCTGTTTCCTGATAACA...,24.342896,1-CCTTATCTB;2-AMCAKMTGKT;4-WTTCCWGGAAW;17-ACCA...
7,8,TCTTATCTTCATGCACCAGACACCTCCAGATAACCGCCAAAAAGTC...,27.684254,1-CCTTATCTB;2-AMCAKMTGKT;3-VTGASTCAB;17-ACCACACC
8,9,CCAGCTGATAAGGAGCTGTCACAAGAAAAAACTGTCAGTCGTCATC...,26.314585,1-CCTTATCTB
9,10,GCTGTGAAACTATGATAATGTCTTATCTGTTCCATGCAGGCGGGGG...,23.901809,1-CCTTATCTB;3-VTGASTCAB;4-WTTCCWGGAAW;13-TCATG...
